<a href="https://colab.research.google.com/github/annarm1/compling/blob/main/Ermolcheva_fine_tuning_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [15]:
# Установка библиотек
!pip install transformers datasets evaluate accelerate gradio -q
!pip install huggingface_hub -q

# Для работы с GPU (проверяем наличие)
import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")


import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GPU доступен: True
Тип GPU: Tesla T4


In [16]:
# 1. Загрузка датасета
from datasets import load_dataset
import numpy as np

dataset = load_dataset("ag_news")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [17]:
# 2. Выбор модели
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
# 3. Токенизация
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [19]:
# 4. Настройка TrainingArguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    logging_steps=500,
)

In [20]:
# 5. Метрики
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [22]:
# 6. Обучение
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.195204,0.176531,0.943158


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.195204,0.176531,0.943158
2,0.138032,0.189841,0.948421
3,0.091949,0.213406,0.946974


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=22500, training_loss=0.15397190534803604, metrics={'train_runtime': 3328.5425, 'train_samples_per_second': 108.155, 'train_steps_per_second': 6.76, 'total_flos': 8558812764910464.0, 'train_loss': 0.15397190534803604, 'epoch': 3.0})

In [23]:
# 7. Оценка
eval_results = trainer.evaluate()
print(f"\nEvaluation results: {eval_results}")


Evaluation results: {'eval_loss': 0.18982674181461334, 'eval_accuracy': 0.9484210526315789, 'eval_runtime': 24.1742, 'eval_samples_per_second': 314.385, 'eval_steps_per_second': 19.649, 'epoch': 3.0}


In [24]:
# Сохранение модели
trainer.save_model('./ag_news_model')
tokenizer.save_pretrained('./ag_news_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./ag_news_model/tokenizer_config.json', './ag_news_model/tokenizer.json')

In [27]:
# 8. Инференс через pipeline
from transformers import pipeline

classifier = pipeline("text-classification", model='./ag_news_model', tokenizer='./ag_news_model')

# Взяла реальные новости с The New-York Times
test_texts = [
    "Is Latin America Ready to Abandon Cuba?",
    "The Moon Turned Blood Red During a Total Lunar Eclipse",
    "Canada wins Paralympic curling gold in one of the more dramatic ways possible",
    "Adidas has made the Champions League ball for 25 years. That could be about to change "
]

#Добавила вот это, чтобы выводил номер лейбла, а сразу категорию
labels = {
    "LABEL_0": "World",
    "LABEL_1": "Sports",
    "LABEL_2": "Business",
    "LABEL_3": "Sci/Tech"
}

for text in test_texts:
    result = classifier(text)[0]
    label = labels[result["label"]]
    score = result["score"]

    print("\nНовость:", text)
    print("Категория:", label)
    print("Score", round(score, 3))

# Модель ошибласть для новости про паралимпийцев, видимо, из-за "Канады". При этом и score у нее самый низкий. Просто fun
# Решила для проверки категории спорт добавить еще одну новость

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Новость: Is Latin America Ready to Abandon Cuba?
Категория: World
Score 0.978

Новость: The Moon Turned Blood Red During a Total Lunar Eclipse
Категория: Sci/Tech
Score 0.991

Новость: Canada wins Paralympic curling gold in one of the more dramatic ways possible
Категория: World
Score 0.908

Новость: Adidas has made the Champions League ball for 25 years. That could be about to change 
Категория: Sports
Score 0.998
